In [1]:
import os
import pandas as pd
import plotly.express as px
import librosa
from tqdm import tqdm

In [2]:
#CONFIG
audio_root = 'train_audio'
taxonomy_csv = 'taxonomy.csv'
output_html = 'birdclef2025_visuals.html'

In [3]:
# Count .ogg files per ID folder
ogg_counts = {}
for folder_name in os.listdir(audio_root):
    folder_path = os.path.join(audio_root, folder_name)
    if os.path.isdir(folder_path):
        ogg_files = [f for f in os.listdir(folder_path) if f.endswith('.ogg')]
        ogg_counts[folder_name] = len(ogg_files)

In [4]:
# Load taxonomy
taxonomy = pd.read_csv(taxonomy_csv)
taxonomy = taxonomy.rename(columns={taxonomy.columns[0]: 'id',
                                    taxonomy.columns[3]: 'name',
                                    taxonomy.columns[4]: 'class'})

In [5]:
# Merge counts with taxonomy
df = pd.DataFrame(list(ogg_counts.items()), columns=['id', 'ogg_count'])
df = df.merge(taxonomy[['id', 'name', 'class']], on='id', how='left')

In [6]:
# Plot 1 - OGG Count per Name
fig1 = px.bar(df.sort_values('ogg_count', ascending=False),
            x='name', y='ogg_count', title='OGG File Count per Bird Name')

In [7]:
# Plot 2 - Average OGG per Name by Class
avg_df = df.groupby('class').agg(avg_ogg_per_name=('ogg_count', 'mean')).reset_index()
fig2 = px.bar(avg_df.sort_values('avg_ogg_per_name', ascending=False),
            x='class', y='avg_ogg_per_name', title='Average OGG Files per Name by Class')

In [8]:
# Save to HTML
from plotly.subplots import make_subplots
from plotly.offline import plot

with open(output_html, 'w') as f:
    f.write(fig1.to_html(full_html=False, include_plotlyjs='cdn'))
    f.write('<hr>')
    f.write(fig2.to_html(full_html=False, include_plotlyjs=False))

In [ ]:
# Measure durations
duration_records = []

print("Scanning audio durations...")
for folder_name in tqdm(os.listdir(audio_root)):
    folder_path = os.path.join(audio_root, folder_name)
    if os.path.isdir(folder_path):
        for file in os.listdir(folder_path):
            if file.endswith('.ogg'):
                file_path = os.path.join(folder_path, file)
                try:
                    duration = librosa.get_duration(path=file_path)
                    duration_records.append({'id': folder_name, 'duration': duration})
                except Exception as e:
                    print(f"Error reading {file_path}: {e}")

duration_df = pd.DataFrame(duration_records)
duration_df = duration_df.merge(taxonomy[['id', 'name', 'class']], on='id', how='left')


In [11]:
# Histogram of all audio durations
fig3 = px.histogram(duration_df, x='duration', nbins=50,
                    title='Distribution of Audio Durations (All Files)',
                    labels={'duration': 'Duration (seconds)'})

# Average duration per name
avg_duration_df = duration_df.groupby('name').agg(avg_duration=('duration', 'mean')).reset_index()
fig4 = px.histogram(avg_duration_df, x='avg_duration', nbins=50,
                    title='Histogram of Average Audio Duration per Bird Name',
                    labels={'avg_duration': 'Avg Duration (seconds)'})


In [12]:
with open(output_html, 'w') as f:
    f.write(fig1.to_html(full_html=False, include_plotlyjs='cdn'))
    f.write('<hr>' + fig2.to_html(full_html=False, include_plotlyjs=False))
    f.write('<hr>' + fig3.to_html(full_html=False, include_plotlyjs=False))
    f.write('<hr>' + fig4.to_html(full_html=False, include_plotlyjs=False))


In [ ]:
# Define bins for OGG file counts
# Adjust bins and labels as needed based on your data's distribution
# Example bins: 0-5, 6-10, 11-20, 21-50, 51-100, 101-200, >200
bin_edges = [0, 5, 10, 20, 50, 100, 200, float('inf')]
bin_labels = ['0-5', '6-10', '11-20', '21-50', '51-100', '101-200', '>200']

# Categorize each species' ogg_count into a bin
# include_lowest=True makes the first interval [0, 5]
df['ogg_bin'] = pd.cut(df['ogg_count'], bins=bin_edges, labels=bin_labels, include_lowest=True)

# Count how many species fall into each bin
species_per_bin = df['ogg_bin'].value_counts().reset_index()
species_per_bin.columns = ['OGG Count Range', 'Number of Species']

# Ensure the bins are sorted correctly for the plot
# pd.cut with labels returns an ordered categorical type, so sort_values works
species_per_bin = species_per_bin.sort_values('OGG Count Range')

# Plot 5 - Number of Species per OGG Count Range
fig5 = px.bar(species_per_bin,
            x='OGG Count Range',
            y='Number of Species',
            title='Number of Species by OGG File Count Range')

# Optional: Add text labels to the bars for exact counts
fig5.update_layout(uniformtext_minsize=8, uniformtext_mode='hide')
fig5.update_traces(texttemplate='%{y}', textposition='outside')

# Display the figure (if running in a notebook)
# fig5.show()

In [15]:
with open(output_html, 'w') as f:
    f.write(fig1.to_html(full_html=False, include_plotlyjs='cdn'))
    f.write('<hr>' + fig2.to_html(full_html=False, include_plotlyjs=False))
    f.write('<hr>' + fig3.to_html(full_html=False, include_plotlyjs=False))
    f.write('<hr>' + fig4.to_html(full_html=False, include_plotlyjs=False))
    # Add the new figure
    f.write('<hr>' + fig5.to_html(full_html=False, include_plotlyjs=False))

In [ ]:
import pandas as pd
import os
import plotly.express as px

# Folder with CSVs
folder = "type_of_call"

# Count rows in each CSV
data = []
for filename in os.listdir(folder):
    if filename.endswith(".csv"):
        path = os.path.join(folder, filename)
        count = len(pd.read_csv(path))
        data.append({"call_type": filename.replace(".csv", ""), "count": count})

# Create DataFrame and sort
df = pd.DataFrame(data)
df = df.sort_values(by="count", ascending=False)

# Generate bar chart
fig = px.bar(df, x="call_type", y="count", title="Call Type Counts (Sorted)", text_auto=True)
fig.update_layout(xaxis_title="Call Type", yaxis_title="Number of Samples")

# Save to HTML
fig.write_html("call_type_chart.html")